# Projekt 01 (basic) — Sentiment-Klassifikation mit einem LSTM

**Modul 09 — NLP 2** · Format: **Jupyter Notebook** (`sentiment_lstm.ipynb`)

In Modul 08 (NLP 1) hast du Texte mit *Zählungen* modelliert: N-Gramme, TF-IDF,
Naive Bayes. Diese Modelle sehen ein Wort als atomares Symbol und die Wortreihenfolge
nur über kurze N-Gramm-Fenster. **Neuronale** Modelle machen zwei Dinge anders:

1. **Embeddings** — jedes Wort wird ein *dichter, gelernter* Vektor (statt eines
   One-Hot-Symbols). Ähnliche Wörter bekommen ähnliche Vektoren.
2. **Rekurrenz (RNN/LSTM)** — das Modell liest den Satz *Wort für Wort* und pflegt
   einen versteckten Zustand, der Kontext über die ganze Sequenz mitträgt.

Du baust hier ein **LSTM** (Long Short-Term Memory), das Filmkritiken & Produktreviews
als *positiv* oder *negativ* klassifiziert, und vergleichst es ehrlich mit einer
Bag-of-Words-Baseline.

> **Viel Anleitung:** Datenaufbereitung, Vokabular, Batching und Trainings-Gerüst sind
> vorgegeben. Deine drei Aufgaben treffen genau die Lernpunkte: das LSTM-Modell
> zusammensetzen, einen Trainingsschritt schreiben und auswerten/vergleichen.


## Was du lernst

- Wie man Text für ein neuronales Modell aufbereitet: **Vokabular**, **Integer-Encoding**,
  **Padding** und **Packing** variabler Sequenzlängen.
- Wie ein **`nn.Embedding` + `nn.LSTM` + Klassifikationskopf** in PyTorch zusammenspielt.
- Warum man bei gepolsterten Batches `pack_padded_sequence` nutzt (das Modell soll das
  Padding *nicht* mitlernen).
- Wie eine **Trainingsschleife** aussieht (Forward → Loss → `backward` → `step`).
- Dass ein LSTM eine starke BoW-Baseline auf *kurzen* Sätzen nur knapp schlägt — und
  *warum* (Ausblick auf Attention/Transformer in Projekt 02 & 03).

## Vorwissen

- Skript Teil 1–3 (Embeddings, RNN/LSTM, Vanishing Gradient).
- PyTorch-Grundlagen aus **Modul 05** (`nn.Module`, Optimizer, `loss.backward()`).


## Setup

Benötigt `torch`, `scikit-learn`, `numpy` (alle in der Repo-`requirements.txt`). Die
erste Code-Zelle **lädt den Datensatz** (UCI *Sentiment Labelled Sentences*, ~85 KB)
nach `datasets/` und cached ihn (per `.gitignore` nicht eingecheckt).

```bash
source ../../../../.venv/bin/activate
jupyter lab      # oder sentiment_lstm.ipynb in VS Code öffnen, Kernel = Repo-.venv
```

Das Training läuft in **wenigen Minuten auf der CPU**. Wenn du eine Apple-GPU (MPS)
oder CUDA hast, nutzt das Notebook sie automatisch.


## Teil A — Daten laden & aufbereiten *(vorgegeben)*

Der Datensatz enthält **3000 kurze Sätze** aus echten Reviews von **IMDb, Amazon und
Yelp**, je zur Hälfte positiv (Label 1) und negativ (Label 0). Beispiel:

> *"A very, very, very slow-moving, aimless movie ..."* → 0
> *"Wow... Loved this place."* → 1

Wir laden, mischen (fester Seed) und teilen in Train/Test.


In [1]:
# ---- Datensatz laden (echte Reviews: IMDb + Amazon + Yelp) -----------------
import os, io, zipfile, urllib.request, random

DATA_DIR = "daten"
os.makedirs(DATA_DIR, exist_ok=True)
FILES = ["imdb_labelled.txt", "amazon_cells_labelled.txt", "yelp_labelled.txt"]
URL = "https://archive.ics.uci.edu/static/public/331/sentiment+labelled+sentences.zip"

if not all(os.path.exists(os.path.join(DATA_DIR, f)) for f in FILES):
    print("Lade Datensatz herunter ...")
    req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
    raw = urllib.request.urlopen(req, timeout=60).read()
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        for name in z.namelist():
            base = os.path.basename(name)
            if base in FILES:
                with z.open(name) as src, open(os.path.join(DATA_DIR, base), "wb") as dst:
                    dst.write(src.read())
    print("Fertig.")
else:
    print("Datensatz bereits vorhanden.")

# Zeilen einlesen: jede Zeile ist "Satz \t Label"
examples = []  # Liste von (text, label)
for f in FILES:
    with open(os.path.join(DATA_DIR, f), encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line or "\t" not in line:
                continue
            text, label = line.rsplit("\t", 1)
            examples.append((text, int(label)))

print(f"{len(examples)} Beispiele geladen.")
print("Beispiel positiv:", next(t for t, y in examples if y == 1)[:60])
print("Beispiel negativ:", next(t for t, y in examples if y == 0)[:60])


Datensatz bereits vorhanden.
3000 Beispiele geladen.
Beispiel positiv: The best scene in the movie was when Gerardo is trying to fi
Beispiel negativ: A very, very, very slow-moving, aimless movie about a distre


In [2]:
# ---- Mischen & Train/Test-Split (fester Seed = reproduzierbar) -------------
random.seed(42)
random.shuffle(examples)

n_test = 600
test_data  = examples[:n_test]
train_data = examples[n_test:]
print(f"Train: {len(train_data)}   Test: {len(test_data)}")

# Klassenbalance pruefen
import numpy as np
tr_y = np.array([y for _, y in train_data])
print(f"Anteil positiv im Train: {tr_y.mean():.2f}")


Train: 2400   Test: 600
Anteil positiv im Train: 0.50


In [3]:
# ---- Tokenisierung & Vokabular --------------------------------------------
import re
from collections import Counter

def tokenize(text):
    # Kleinschreibung + Woerter/Zahlen als Tokens (Satzzeichen fallen weg)
    return re.findall(r"[a-z0-9']+", text.lower())

# Vokabular NUR aus den Trainingsdaten aufbauen (kein Blick auf Test!)
counter = Counter(tok for text, _ in train_data for tok in tokenize(text))
MIN_FREQ = 2  # seltene Woerter (Freq 1) -> <unk>, haelt das Vokabular klein

PAD, UNK = "<pad>", "<unk>"
itos = [PAD, UNK] + [w for w, c in counter.most_common() if c >= MIN_FREQ]
stoi = {w: i for i, w in enumerate(itos)}
PAD_IDX, UNK_IDX = stoi[PAD], stoi[UNK]
VOCAB_SIZE = len(itos)
print(f"Vokabulargroesse: {VOCAB_SIZE}")

def encode(text):
    """Satz -> Liste von Integer-IDs (unbekannte Woerter -> <unk>)."""
    return [stoi.get(tok, UNK_IDX) for tok in tokenize(text)]

print("Encoding-Beispiel:", encode("loved this place")[:10])


Vokabulargroesse: 1943
Encoding-Beispiel: [255, 10, 42]


In [4]:
# ---- PyTorch-Dataset & DataLoader mit Padding ------------------------------
import torch
from torch.utils.data import Dataset, DataLoader

device = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
print("Device:", device)

class SentimentDataset(Dataset):
    def __init__(self, data):
        self.data = [(encode(t), y) for t, y in data if len(encode(t)) > 0]
    def __len__(self):
        return len(self.data)
    def __getitem__(self, i):
        ids, y = self.data[i]
        return torch.tensor(ids, dtype=torch.long), torch.tensor(y, dtype=torch.float)

def collate(batch):
    """Polstert einen Batch auf gleiche Laenge und liefert die echten Laengen mit.
    Die Laengen brauchen wir spaeter fuer pack_padded_sequence."""
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    maxlen = lengths.max().item()
    padded = torch.full((len(seqs), maxlen), PAD_IDX, dtype=torch.long)
    for i, s in enumerate(seqs):
        padded[i, :len(s)] = s
    return padded, lengths, torch.stack(labels)

BATCH = 32
train_loader = DataLoader(SentimentDataset(train_data), batch_size=BATCH,
                          shuffle=True, collate_fn=collate)
test_loader  = DataLoader(SentimentDataset(test_data), batch_size=BATCH,
                          shuffle=False, collate_fn=collate)

xb, lb, yb = next(iter(train_loader))
print("Batch-Form:", xb.shape, "Laengen[:5]:", lb[:5].tolist())


Device: mps
Batch-Form: torch.Size([32, 22]) Laengen[:5]: [12, 10, 10, 9, 9]


### Aufgabe 1 — Das LSTM-Modell zusammensetzen

Baue den Klassifikator. Die Architektur:

$$\text{IDs} \;\xrightarrow{\text{Embedding}}\; \mathbf{e}_1\dots\mathbf{e}_T
\;\xrightarrow{\text{LSTM}}\; \mathbf{h}_1\dots\mathbf{h}_T \;\longrightarrow\;
\mathbf{h}_T \xrightarrow{\text{Linear}} \text{Logit}$$

Zu tun in `forward` (drei mit `# TODO` markierte Stellen):

1. **Embedding:** `emb = self.embedding(x)` → Form `(Batch, T, embed_dim)`.
2. **Packen + LSTM:** Wir wollen, dass das LSTM das **Padding ignoriert**. Dazu die
   Embeddings mit `pack_padded_sequence(emb, lengths, batch_first=True,
   enforce_sorted=False)` verpacken und durch `self.lstm` schicken. Das LSTM gibt
   `(output, (h_n, c_n))` zurück — wir brauchen **`h_n`**, den letzten versteckten
   Zustand (Form `(1, Batch, hidden_dim)`).
3. **Kopf:** aus `h_n` den Batch-Vektor `h_n[-1]` (Form `(Batch, hidden_dim)`) nehmen,
   Dropout anwenden und durch `self.fc` auf **einen** Logit pro Beispiel abbilden;
   mit `.squeeze(1)` die letzte Dimension entfernen.

> Warum `h_n` und nicht `output`? `h_n` ist der Zustand *nach dem letzten echten Wort*
> jeder Sequenz — genau die Zusammenfassung des ganzen Satzes, die wir klassifizieren.


In [5]:
# ---- AUFGABE 1: LSTM-Klassifikator ----------------------------------------
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=64, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        # 1) Embedding-Lookup
        emb = self.embedding(x)                       # (B, T, E)
        # 2) Padding verpacken, damit das LSTM es ignoriert
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True,
                                      enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)               # h_n: (1, B, H)
        # 3) Klassifikationskopf
        h = self.dropout(h_n[-1])                      # (B, H)
        return self.fc(h).squeeze(1)                   # (B,)

model = LSTMClassifier(VOCAB_SIZE, pad_idx=PAD_IDX).to(device)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameter: {n_params:,}")


LSTMClassifier(
  (embedding): Embedding(1943, 64, padding_idx=0)
  (lstm): LSTM(64, 64, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)
Parameter: 157,697


### Aufgabe 2 — Ein Trainingsschritt

Die Schleifen-Struktur (Epochen, Batches, Auswertung) ist vorgegeben. Fülle in
`train_one_epoch` den **Kern eines Trainingsschritts** aus (vier `# TODO`-Zeilen):

1. `optimizer.zero_grad()` — alte Gradienten löschen.
2. **Forward:** `logits = model(xb, lengths)`.
3. **Loss:** `loss = criterion(logits, yb)` (Binary Cross-Entropy auf den Logits).
4. **Backward + Update:** `loss.backward()`, dann `optimizer.step()`.

Wir nutzen `BCEWithLogitsLoss` — die kombiniert Sigmoid + Binary-Cross-Entropy
numerisch stabil, deshalb gibt das Modell rohe **Logits** (keine Wahrscheinlichkeit)
aus.


In [6]:
# ---- Auswertungs-Helfer (vorgegeben) --------------------------------------
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for xb, lengths, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb, lengths)
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == yb).sum().item()
        total += yb.size(0)
    return correct / total


In [7]:
# ---- AUFGABE 2: Trainingsschritt + Trainingslauf --------------------------
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    for xb, lengths, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()                 # 1) Gradienten loeschen
        logits = model(xb, lengths)           # 2) Forward
        loss = criterion(logits, yb)          # 3) Loss
        loss.backward()                       # 4a) Backprop
        optimizer.step()                      # 4b) Update
        total_loss += loss.item() * yb.size(0)
    return total_loss / len(loader.dataset)

EPOCHS = 8
for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch(model, train_loader)
    tr_acc = evaluate(model, train_loader)
    te_acc = evaluate(model, test_loader)
    print(f"Epoche {epoch:2d} | Loss {loss:.3f} | Train-Acc {tr_acc:.3f} | Test-Acc {te_acc:.3f}")


Epoche  1 | Loss 0.674 | Train-Acc 0.726 | Test-Acc 0.655


Epoche  2 | Loss 0.553 | Train-Acc 0.807 | Test-Acc 0.750


Epoche  3 | Loss 0.423 | Train-Acc 0.892 | Test-Acc 0.775


Epoche  4 | Loss 0.317 | Train-Acc 0.925 | Test-Acc 0.772


Epoche  5 | Loss 0.211 | Train-Acc 0.962 | Test-Acc 0.793


Epoche  6 | Loss 0.147 | Train-Acc 0.974 | Test-Acc 0.770


Epoche  7 | Loss 0.096 | Train-Acc 0.982 | Test-Acc 0.785


Epoche  8 | Loss 0.071 | Train-Acc 0.993 | Test-Acc 0.788


### Aufgabe 3 — Baseline-Vergleich & eigene Sätze

**(a)** Die Bag-of-Words-Baseline (`CountVectorizer` + `LogisticRegression` aus
scikit-learn, wie in Modul 08) ist vorgegeben. Trage in der Markdown-Reflexion unten
ein, wie sie im Vergleich zum LSTM abschneidet.

**(b)** Fülle `predict_sentiment` aus (zwei `# TODO`): einen einzelnen Satz encoden,
als Batch der Größe 1 durchs Modell schicken, Sigmoid anwenden und die
Wahrscheinlichkeit zurückgeben. Teste eigene Sätze.


In [8]:
# ---- (a) Bag-of-Words-Baseline (vorgegeben, zum Vergleich) -----------------
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

vec = CountVectorizer(tokenizer=tokenize, token_pattern=None)
Xtr = vec.fit_transform([t for t, _ in train_data])
Xte = vec.transform([t for t, _ in test_data])
ytr = [y for _, y in train_data]
yte = [y for _, y in test_data]

clf = LogisticRegression(max_iter=1000)
clf.fit(Xtr, ytr)
bow_acc = clf.score(Xte, yte)
print(f"BoW + LogReg Test-Acc: {bow_acc:.3f}")
print(f"LSTM         Test-Acc: {evaluate(model, test_loader):.3f}")


BoW + LogReg Test-Acc: 0.860
LSTM         Test-Acc: 0.788


In [9]:
# ---- (b) AUFGABE 3: eigene Saetze klassifizieren --------------------------
@torch.no_grad()
def predict_sentiment(text):
    model.eval()
    ids = encode(text)
    if not ids:
        return 0.5
    # Batch der Groesse 1 bauen
    x = torch.tensor([ids], dtype=torch.long).to(device)      # (1, T)
    lengths = torch.tensor([len(ids)])
    logit = model(x, lengths)                                  # (1,)
    prob = torch.sigmoid(logit).item()
    return prob

for s in ["I absolutely loved this movie, fantastic!",
          "Terrible product, broke after one day.",
          "The food was okay, nothing special."]:
    p = predict_sentiment(s)
    print(f"[{'POS' if p>0.5 else 'NEG'} p={p:.2f}]  {s}")


[POS p=1.00]  I absolutely loved this movie, fantastic!
[NEG p=0.01]  Terrible product, broke after one day.
[NEG p=0.01]  The food was okay, nothing special.


## Reflexion (kurz, schriftlich)

Beantworte in eigenen Worten:

1. **LSTM vs. BoW — der Aha-Moment:** Wie groß ist der Abstand? Überraschung: die
   simple BoW-Baseline **schlägt** das LSTM hier deutlich (~0.86 vs. ~0.79). Das ist
   *kein* Fehler, sondern die zentrale Lehre dieses Projekts: neuronale Modelle sind
   *datenhungrig*. Bei nur ~2400 sehr *kurzen* Trainingssätzen trägt die reine
   *Wortpräsenz* („loved", „terrible", „waste") schon fast die ganze Information, und
   ein lineares Modell über tausende Bag-of-Words-Features generalisiert darauf robust.
   Das LSTM mit gelernten Embeddings hat viel mehr Kapazität, **overfittet** aber auf
   der kleinen Datenmenge (siehe Punkt 3). Die Wortreihenfolge, die das LSTM zusätzlich
   modelliert, zahlt sich erst bei *mehr* und *längeren* Texten aus (Negation über
   Distanz, „not … good"). Merke: „mehr Modell" ≠ „mehr Genauigkeit", wenn die Daten
   knapp sind.

2. **Padding & Packing:** Was würde passieren, wenn wir das Padding *nicht* mit
   `pack_padded_sequence` ausblenden, sondern die `<pad>`-Tokens einfach mit durchs
   LSTM laufen ließen?

3. **Overfitting:** Train-Acc steigt gegen 1.0, Test-Acc bleibt darunter. Woran siehst
   du Overfitting, und welche zwei Stellschrauben (im Code sichtbar) wirken dagegen?

4. **Ausblick:** Das LSTM liest streng *sequenziell* und muss den ganzen Satz in **einen**
   Vektor $\mathbf{h}_T$ pressen. Welches Problem ergibt sich daraus bei langen
   Sequenzen — und wie greift **Attention** (Projekt 02) das an?

> **Referenzgrößen** (fester Seed): LSTM Test-Acc ~0.78–0.80 (Train-Acc ~0.99 →
> deutliches Overfitting), BoW+LogReg ~0.85–0.86. Genaue Zahlen schwanken je nach
> Hardware leicht — dass die BoW-Baseline hier *gewinnt*, ist der eigentliche Lernpunkt.
